# 1. set up GPU

In [1]:
import sys
#sys.path.insert(0,'...Path where the corresponding environment is...')

In [2]:
from tensorflow.python.util import deprecation
deprecation._PRINT_DEPRECATION_WARNINGS = False   #ignore FutureWarning

import tensorflow as tf
tf.__version__   #version 1.9

'2.10.0'

# 2. import libraries, data preparation

In [3]:
import skimage
from tqdm import tqdm
from skimage import io

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import img_to_array, load_img, array_to_img

In [4]:
disease_class = {'Atelectasis': [],
                 'Cardiomegaly': [],
                 'Effusion': [],
                 'Infiltration': [],
                 'Mass': [],
                 'Nodule': [],
                 'Pneumonia': [],
                 'Pneumothorax': [],
                 'Consolidation': [],
                 'Edema': [],
                 'Emphysema': [],
                 'Fibrosis': [],
                 'Pleural_Thickening': [],
                 'Hernia': [],
                 'No Finding':[]}

#import labels of the images
data_ref = pd.read_csv("NIH Dataset/Data_Entry_2017.csv")
pd.options.mode.chained_assignment = None        #ignore the SettingWithCopyWarning


for i in tqdm(range(len(data_ref))):
    #only record single-labelled images
    
    if "|" not in data_ref['Finding Labels'][i]:
        disease_class[data_ref['Finding Labels'][i]].append(data_ref['Image Index'][i])

100%|███████████████████████████████████████████████████████████████████████| 112120/112120 [00:02<00:00, 49067.58it/s]


In [24]:
lengths = []
for key in disease_class.keys():
    lengths.append(len(disease_class[key]))

In [25]:
lengths = np.array(lengths) * 7
lengths = np.where(lengths > 4000, 4000, lengths)

In [26]:
np.sum(lengths)

55024

In [28]:
import os
for folder in os.listdir('NIH Dataset/Images_NOISED/train'):
    os.makedirs('Generated Images/ACGAN/' + folder)
    os.makedirs('Generated Images/WGAN/' + folder)

In [5]:
# inject noise into every images of the original dataset

# number = 91324
# img_names = []

# for dis in disease_class.keys():
#     num = round(number/91324*len(disease_class[dis]))
#     for i in range(num):
#         img_names.append(disease_class[dis][i])

# 3. Noise Generation Function

In [6]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [11]:
def gen_noise(filename, cls, mode):
    #load dataset and resize every image to 112 x 112
    img = img_to_array(load_img(img_path + filename, target_size=(112,112)))
    if mode == None:
        gimg = img
        mode = ''
        
    elif mode == 'gaussian' or mode == 'poisson':
        img = img / 255.0
        gimg = skimage.util.random_noise(img, mode = mode) #add noise to every image
        
    elif mode[0] == 'r':
        img = img / 255.0
        gimg = skimage.transform.rotate(img, int(mode[1 : ]))
            
    gimg = array_to_img(gimg)
    gimg.save("NIH Dataset/Images_NOISED/train/"+cls+"/"+mode+filename)

# 3. Save single copies of noise-augmented images to directory

In [13]:
img_path = 'NIH Dataset/imgs/'
generated = {1000, 200, 300, 100, 1200, 150, 100, 390, 3000, 500, 9610, 500, 200, 50, 400}
generated = {'Atelectasis': 1000,
                 'Cardiomegaly': 200,
                 'Consolidation': 300,
                 'Edema': 100,
                 'Effusion': 1200,
                 'Emphysema': 150,
                 'Fibrosis': 100,
                 'Hernia': 390,
                 'Infiltration': 3000,
                 'Mass': 500,
                 'No Finding': 9610,
                 'Nodule': 500,
                 'Pleural_Thickening': 200,
                 'Pneumonia': 50,
                 'Pneumothorax':400}

idx = 0
augments = ['gaussian', 'poisson', 'r20', 'r40', 'r60', 'r80', 'r100', 'r120', 'r140', 'r160', 'r180', 'r200', 'r220', 'r240', 'r260', 'r280']
# poisson noise performs the best
for key in disease_class.keys():
    images = disease_class[key]
    idx = 0
    for i in tqdm(range(len(images))):
        if len(images) > 2000:
            if i >= 2100:
                break
            gen_noise(images[i], key, None)
        else:
            if i == 0:
                for j in range(len(images)):
                    gen_noise(images[j], key, None)
                    idx += 1
            for augment in augments:
                if idx >= 2100:
                    break
                gen_noise(images[i], key, augment)
                idx += 1

  3%|██▋                                                                          | 2100/60361 [01:39<46:05, 21.07it/s]


In [14]:
import os
folders = os.listdir('NIH Dataset/Images_NORMAL/train')
# for folder in folders:
#     os.makedirs('NIH Dataset/Images_NORMAL/valid/' + folder)
#     os.makedirs('NIH Dataset/Images_NOISED/valid/' + folder)
#     os.makedirs('NIH Dataset/Images_ACGAN/valid/' + folder)
#     os.makedirs('NIH Dataset/Images_WCGAN/valid/' + folder)

In [15]:
import shutil

In [16]:
for folder in folders:
#     for idx, image in enumerate(os.listdir('NIH Dataset/Images_NORMAL/train/' + folder)):
#         if idx == 20:
#             break
#         shutil.move('NIH Dataset/Images_NORMAL/train/' + folder + '/' + image, 'NIH Dataset/Images_NORMAL/valid/' + folder + '/' + image)
    for idx, image in enumerate(os.listdir('NIH Dataset/Images_NOISED/train/' + folder)):
        if idx == 100:
            break
        shutil.move('NIH Dataset/Images_NOISED/train/' + folder + '/' + image, 'NIH Dataset/Images_NOISED/valid/' + folder + '/' + image)
#     for idx, image in enumerate(os.listdir('NIH Dataset/Images_ACGAN/train/' + folder)):
#         if idx == 20:
#             break
#         shutil.move('NIH Dataset/Images_ACGAN/train/' + folder + '/' + image, 'NIH Dataset/Images_ACGAN/valid/' + folder + '/' + image)
#     for idx, image in enumerate(os.listdir('NIH Dataset/Images_WCGAN/train/' + folder)):
#         if idx == 20:
#             break
#         shutil.move('NIH Dataset/Images_WCGAN/train/' + folder + '/' + image, 'NIH Dataset/Images_WCGAN/valid/' + folder + '/' + image)